# Concept robustness and cross-model evaluation

Evaluate saved concepts from five independent GPT-5 calls for Zeisel and Pollen under the original, conservative, and cell-identity-aware prompts. Recompute clustering, interpretability, and matched-gene-set similarity, evaluate saved consensus concepts, and compare deposited GPT-5 and Claude coherence scores. No model training or API calls are required for the default workflow.

## Prerequisites

- Install scConcept and prepare processed datasets using Tutorials 0–3.
- Generate the required concepts first, or use the deposited concept files in `Tutorial2/Concept`.
- Configure the input paths below. Processed `.mat` files use `Datasets/`; additional clinical, annotation, or plotting inputs are required by the corresponding analysis.

This tutorial starts from saved concepts. Run cells from top to bottom; preprocessing, topic training, and concept generation are not repeated. The shared input helpers are in `Tutorial2/tutorial_utils.py`.


In [ ]:
from pathlib import Path
import os
import json
import hashlib
import platform
import importlib.metadata
from collections import Counter
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy import sparse
from scipy.optimize import linear_sum_assignment
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, normalized_mutual_info_score
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

# =========================================================
# Locate the installed scConcept repository and tutorial inputs
# =========================================================
import os
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scConcept.py").is_file() and (p / "Tutorial2").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the scConcept repository or one of its subdirectories.")
TUTORIAL_DIR = PROJECT_ROOT / "Tutorial2"
for directory in [PROJECT_ROOT, TUTORIAL_DIR]:
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from tutorial_utils import (ScConcept, SingleCellDataset, configure_inputs,
    input_path, dataset_path, load_mat_test, load_api_dataset, load_saved_concepts,
    score_saved_concepts, matlab_string, runtime_paths,
    compute_coherence_from_gene_lists, TD_eva_topics, read_pathways, interpretability)
PACKAGE_DATA_DIR = configure_inputs()
REPO_DIR = PROJECT_ROOT
CONCEPT_DIR = TUTORIAL_DIR / "Concept"

# Dataset paths can be overridden in Inputs/data_paths.local.json.
INPUT_DIR = TUTORIAL_DIR / "Inputs/robustness"
MAT_DIR = PACKAGE_DATA_DIR
GMT_PATHS = {species: input_path(f"gmt.{species}", f"msigdb.v2024.1.{species}.symbols.gmt") for species in ["Hs", "Mm"]}
OUTPUT_DIR = TUTORIAL_DIR / "outputs/robustness"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS = [1, 2, 3, 4, 5]
DATASETS = {"Zeisel": "zeisel", "Pollen": "pollen"}
SETTINGS = {"Original": "original", "Conservative": "conservative", "Cell-identity": "cellidentity"}
TOP_K = 100
MATCHING_TOP_K = 100
INFO_METRIC = "AMI"
RUN_PLOTS = True
RUN_NEW_CLAUDE_SCORING = False
STABILITY_TC_BACKGROUND = {"Zeisel": "Mm", "Pollen": "Mm"}
PROMPT_TC_BACKGROUND = {"Zeisel": "Mm", "Pollen": "Hs"}
PROMPTS = json.loads((INPUT_DIR / "prompts.json").read_text())


## 1. Scoring and evaluation

The five calls used the same generation seed, 1. Run identifiers distinguish saved outputs; they are not five different generation seeds. Full prompts are stored in `Inputs/robustness/prompts.json`.

Score each concept using the mean gene-wise z-score of its first 100 listed genes, with standard deviation plus 1e-6. Assign cells to their highest-scoring concept. Calculate TC and TD from the first 10 genes using the original fixed-denominator definitions. Report actual NMI and AMI separately. `INFO_METRIC="AMI"` reproduces the information metric used in the original plots and labels it correctly; choose `"NMI"` to plot normalized mutual information.

The stability/consensus panel uses mouse MSigDB for both datasets. The prompt-sensitivity panels use mouse MSigDB for Zeisel and human MSigDB for Pollen. Both TC backgrounds are exported for every evaluated concept set.

The helpers below retain the reported TC/TD definitions. They are evaluated from each current concept set, rather than reusing cached annotations from `ScConcept.evaluation()`.


In [ ]:


def load_expression(dataset):
    """Load the prepared dataset through the same loader used by ScConcept."""
    handler = load_api_dataset(dataset_path(DATASETS[dataset]))
    X = handler.test_data.detach().cpu().numpy().astype(float)
    genes = [str(g) for g in handler.gene_names]
    labels = np.asarray([matlab_string(v) for v in handler.test_labels.reshape(-1)])
    if X.shape != (len(labels), len(genes)) or len(set(genes)) != len(genes):
        raise ValueError(f"Expression/label/vocabulary mismatch: {dataset}")
    # Standardize once per dataset; each saved run is evaluated independently.
    z = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-6)
    return z, genes, labels

def evaluate_concepts(concepts, z, genes, labels, pathways):
    """Apply ScConcept's annotation rule to cached gene z-scores for each saved run."""
    index = {g: i for i, g in enumerate(genes)}
    scores = np.zeros((len(z), len(concepts)))
    matches = []
    for i, concept in enumerate(concepts):
        requested = concept["genes"][:TOP_K]
        matched = [g for g in requested if g in index]
        if matched:
            scores[:, i] = z[:, [index[g] for g in matched]].mean(axis=1)
        matches.append(dict(concept=concept["name"], requested=requested, matched=matched))
    # Use a fresh assignment for every run and prompt.
    prediction = scores.argmax(axis=1)
    table = pd.crosstab(pd.Series(labels), pd.Series(prediction))
    metrics = dict(n_cells=len(labels), Concept_num=len(concepts), Purity=float(table.max(axis=0).sum() / len(labels)),
        ARI=adjusted_rand_score(labels, prediction), NMI=normalized_mutual_info_score(labels, prediction),
        AMI=adjusted_mutual_info_score(labels, prediction))
    for species, background in pathways.items():
        values = interpretability(concepts, genes, background, 10)
        metrics[f"TC_{species}"] = values["TC"]
        metrics["TD"] = values["TD"]
        metrics["TD_unique_fraction"] = values["TD_unique_fraction"]
    return metrics, scores, matches


## 2. Evaluate saved runs and consensus concepts

Read all 30 run-specific concept files and the two saved consensus files. Save performance, gene matches, and cell assignments for each set. Summaries use sample standard deviation across the five calls.


In [ ]:
pathways = {species: read_pathways(path) for species, path in GMT_PATHS.items()}
# Evaluate saved gene lists; do not regenerate concepts or call an LLM.
all_concepts = {}
rows = []
concept_rows = []
for dataset in DATASETS:
    z, genes, labels = load_expression(dataset)
    all_concepts[dataset] = {}
    for setting, filename_setting in SETTINGS.items():
        all_concepts[dataset][setting] = {}
        for run in RUNS:
            path = CONCEPT_DIR / f"GPT5_{dataset}_concepts_{filename_setting}_run{run}.json"
            concepts = load_saved_concepts(path)
            all_concepts[dataset][setting][run] = concepts
            metrics, scores, matches = evaluate_concepts(concepts, z, genes, labels, pathways)
            rows.append(dict(dataset=dataset, setting=setting, run=run, **metrics))
            out = OUTPUT_DIR / dataset / filename_setting / f"run{run}"
            out.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(out / "concept_scores.npz", scores=scores, names=np.asarray([c["name"] for c in concepts]))
            pd.DataFrame(dict(cell_index=np.arange(len(labels)), cell_type=labels, concept=np.asarray([c["name"] for c in concepts])[scores.argmax(axis=1)])).to_csv(out / "assignments.csv", index=False)
            (out / "gene_matches.json").write_text(json.dumps(matches, indent=2))
            for i, c in enumerate(concepts):
                concept_rows.append(dict(dataset=dataset, setting=setting, run=run, concept_index=i, name=c["name"], genes=json.dumps(c["genes"]), n_genes=len(c["genes"])))
    concepts = load_saved_concepts(CONCEPT_DIR / f"GPT5_{dataset}_concepts_consensus.json")
    metrics, scores, matches = evaluate_concepts(concepts, z, genes, labels, pathways)
    rows.append(dict(dataset=dataset, setting="Consensus", run=np.nan, **metrics))
    out = OUTPUT_DIR / dataset / "consensus"
    out.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out / "concept_scores.npz", scores=scores, names=np.asarray([c["name"] for c in concepts]))
    (out / "gene_matches.json").write_text(json.dumps(matches, indent=2))
performance = pd.DataFrame(rows)
performance.to_csv(OUTPUT_DIR / "concept_performance.csv", index=False)
pd.DataFrame(concept_rows).to_csv(OUTPUT_DIR / "run_concepts.csv", index=False)
metric_columns = ["ARI", "NMI", "AMI", "TC_Hs", "TC_Mm", "TD", "Concept_num"]
performance[performance.setting != "Consensus"].groupby(["dataset", "setting"])[metric_columns].agg(["mean", "std"]).to_csv(OUTPUT_DIR / "performance_summary.csv")
display(performance)


## 3. Plot run stability and prompt sensitivity

Retain the original panel layouts, colors, individual-run points, sample-standard-deviation error bars, and consensus markers. All plotted values are read from the recomputed results.


In [ ]:
if RUN_PLOTS:
    datasets = {}
    for dataset in DATASETS:
        view = performance[performance.dataset == dataset].copy()
        view["TC"] = view[f"TC_{STABILITY_TC_BACKGROUND[dataset]}"]
        datasets[dataset] = {"runs": view[view.setting == "Original"].sort_values("run"),
                             "consensus": view[view.setting == "Consensus"].iloc[0].to_dict()}
    colors = {'ARI': '#4C72B0', INFO_METRIC: '#55A868', 'TC': '#C44E52', 'TD': '#8172B2'}
    consensus_color = '#D97706'
    out_dir = str(OUTPUT_DIR)
    os.makedirs(out_dir, exist_ok=True)
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 12, 'xtick.labelsize': 11, 'ytick.labelsize': 10, 'axes.linewidth': 1.0, 'pdf.fonttype': 42, 'ps.fonttype': 42})
    
    def draw(ax, runs, consensus, metrics):
        xpos = np.arange(len(metrics), dtype=float)
        run_offset = -0.13
        mean_offset = 0.0
        consensus_offset = 0.17
        jitter = np.linspace(-0.055, 0.055, len(runs))
        for (x, metric) in zip(xpos, metrics):
            y = runs[metric].to_numpy(dtype=float)
            color = colors[metric]
            consensus_value = consensus[metric]
            ax.scatter(x + run_offset + jitter, y, s=32, facecolors='white', edgecolors=color, linewidths=1.3, zorder=3)
            ax.errorbar(x + mean_offset, y.mean(), yerr=y.std(ddof=1), fmt='o', color=color, markersize=6, capsize=3.5, elinewidth=1.4, markeredgewidth=1.2, zorder=4)
            ax.scatter(x + consensus_offset, consensus_value, marker='D', s=52, facecolors=consensus_color, edgecolors='white', linewidths=0.7, zorder=5)
        ax.set_xticks(xpos)
        ax.set_xticklabels(metrics)
        ax.set_xlim(-0.4, len(metrics) - 1 + 0.43)
        ax.set_ylim(0, 1.12)
        ax.set_yticks(np.arange(0, 1.01, 0.2))
        ax.set_ylabel('Score')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(direction='out', length=4, width=1)
    (fig, axes) = plt.subplots(2, 2, figsize=(7.2, 6.0), sharey=True)
    metric_groups = [['ARI', INFO_METRIC], ['TC', 'TD']]
    panel_letters = [['a', 'b'], ['c', 'd']]
    for (row, (dataset_name, data)) in enumerate(datasets.items()):
        for (col, metrics) in enumerate(metric_groups):
            ax = axes[row, col]
            draw(ax, runs=data['runs'], consensus=data['consensus'], metrics=metrics)
            ax.set_title(dataset_name, fontsize=12, pad=12)
            if col == 1:
                ax.set_ylabel('')
    legend_handles = [Line2D([], [], marker='o', linestyle='none', markerfacecolor='white', markeredgecolor='#555555', markeredgewidth=1.3, markersize=6, label='Individual runs'), axes[0, 0].errorbar([], [], yerr=[], fmt='o', color='#555555', markersize=6, capsize=3.5, elinewidth=1.4, label='Mean ± SD'), Line2D([], [], marker='D', linestyle='none', markerfacecolor=consensus_color, markeredgecolor='white', markeredgewidth=0.7, markersize=7, label='Consensus')]
    fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.015), ncol=3, frameon=False, fontsize=10, handletextpad=0.6, columnspacing=1.8)
    fig.subplots_adjust(left=0.11, right=0.97, bottom=0.15, top=0.92, wspace=0.28, hspace=0.48)
    output_stem = os.path.join(out_dir, 'Zeisel_Pollen_LLM_run_stability_with_consensus')
    fig.savefig(output_stem + '.pdf', bbox_inches='tight')
    fig.savefig(output_stem + '.png', dpi=600, bbox_inches='tight')
    plt.show()


In [ ]:
if RUN_PLOTS:
    for dataset in DATASETS:
        view = performance[(performance.dataset == dataset) & (performance.setting != "Consensus")].copy()
        view["TC"] = view[f"TC_{PROMPT_TC_BACKGROUND[dataset]}"]
        data = {setting: view[view.setting == setting].sort_values("run").to_dict(orient="list") for setting in SETTINGS}
        prompts = ['Original', 'Conservative', 'Cell-identity']
        colors = {'Original': '#4C72B0', 'Conservative': '#55A868', 'Cell-identity': '#C44E52'}
        out_dir = str(OUTPUT_DIR / dataset)
        os.makedirs(out_dir, exist_ok=True)
        plt.rcParams.update({'font.family': 'Arial', 'font.size': 11, 'axes.labelsize': 14, 'xtick.labelsize': 11, 'ytick.labelsize': 12, 'axes.linewidth': 1.0, 'pdf.fonttype': 42, 'ps.fonttype': 42})
        print('\nMean ± SD')
        print('=' * 75)
        for metric in ['ARI', INFO_METRIC, 'TC', 'TD', 'Concept_num']:
            print(f'\n{metric}')
            for prompt in prompts:
                y = np.array(data[prompt][metric])
                print(f'{prompt:15s}: {y.mean():.4f} ± {y.std(ddof=1):.4f}')
        
        def draw_metric(ax, metric):
            xpos = np.arange(len(prompts))
            jitter = np.linspace(-0.06, 0.06, 5)
            for (x, prompt) in zip(xpos, prompts):
                y = np.array(data[prompt][metric])
                c = colors[prompt]
                ax.scatter(x + jitter, y, s=38, facecolor='white', edgecolor=c, linewidth=1.4, zorder=3)
                ax.errorbar(x, y.mean(), yerr=y.std(ddof=1), fmt='o', color=c, markersize=7, capsize=4, elinewidth=1.5, zorder=4)
            ax.set_xticks(xpos)
            ax.set_xticklabels(['Original', 'Conservative', 'Cell-identity'], rotation=25, ha='right')
            ax.set_ylim(0, 1.05)
            ax.set_yticks(np.arange(0, 1.01, 0.2))
            ax.set_ylabel(metric)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(direction='out', length=4, width=1)
        (fig, axes) = plt.subplots(1, 4, figsize=(10.0, 3.2))
        for (ax, metric) in zip(axes, ['ARI', INFO_METRIC, 'TC', 'TD']):
            draw_metric(ax, metric)
        plt.subplots_adjust(left=0.07, right=0.99, bottom=0.28, top=0.96, wspace=0.42)
        plt.savefig(os.path.join(out_dir, f'{dataset}_prompt_sensitivity.pdf'), bbox_inches='tight')
        plt.show()
        (fig, ax) = plt.subplots(figsize=(3.8, 3.2))
        xpos = np.arange(len(prompts))
        jitter = np.linspace(-0.06, 0.06, 5)
        for (x, prompt) in zip(xpos, prompts):
            y = np.array(data[prompt]['Concept_num'])
            c = colors[prompt]
            ax.scatter(x + jitter, y, s=42, facecolor='white', edgecolor=c, linewidth=1.4, zorder=3)
            ax.errorbar(x, y.mean(), yerr=y.std(ddof=1), fmt='o', color=c, markersize=7, capsize=4, elinewidth=1.5, zorder=4)
        ax.set_xticks(xpos)
        ax.set_xticklabels(['Original', 'Conservative', 'Cell-identity'], rotation=25, ha='right')
        ax.set_ylabel('Number of concepts')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(direction='out', length=4, width=1)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'{dataset}_prompt_concept_number.pdf'), bbox_inches='tight')
        plt.show()


## 4. Match concepts between runs

Use Hungarian one-to-one matching to maximize Jaccard overlap of normalized top-100 gene sets. Compare all ten run pairs within each dataset/prompt combination. Report unmatched concept counts separately; unmatched concepts do not enter the mean matched Jaccard.


In [ ]:
def gene_jaccard(concept1, concept2, top_k=20):
    genes1 = {str(g).strip().upper() for g in concept1['genes'][:top_k]}
    genes2 = {str(g).strip().upper() for g in concept2['genes'][:top_k]}
    union = genes1 | genes2
    if len(union) == 0:
        return 0.0
    return len(genes1 & genes2) / len(union)

def match_concepts_between_runs(concepts1, concepts2, top_k=20):
    n1 = len(concepts1)
    n2 = len(concepts2)
    similarity = np.zeros((n1, n2))
    for (i, c1) in enumerate(concepts1):
        for (j, c2) in enumerate(concepts2):
            similarity[i, j] = gene_jaccard(c1, c2, top_k=top_k)
    (row_idx, col_idx) = linear_sum_assignment(-similarity)
    matched_rows = []
    for (i, j) in zip(row_idx, col_idx):
        matched_rows.append({'concept_run1': concepts1[i]['name'], 'concept_run2': concepts2[j]['name'], 'jaccard': similarity[i, j]})
    matched_df = pd.DataFrame(matched_rows)
    matched_run1 = set(row_idx)
    matched_run2 = set(col_idx)
    unmatched_run1 = [concepts1[i]['name'] for i in range(n1) if i not in matched_run1]
    unmatched_run2 = [concepts2[j]['name'] for j in range(n2) if j not in matched_run2]
    summary = {'n_concepts_run1': n1, 'n_concepts_run2': n2, 'n_matched': len(row_idx), 'n_unmatched_run1': len(unmatched_run1), 'n_unmatched_run2': len(unmatched_run2), 'mean_matched_jaccard': matched_df['jaccard'].mean(), 'median_matched_jaccard': matched_df['jaccard'].median()}
    return (matched_df, summary, unmatched_run1, unmatched_run2, similarity)


In [ ]:
# Compare each of the ten run pairs within the same dataset and prompt.
matched_rows = []
summary_rows = []
for dataset in DATASETS:
    for setting in SETTINGS:
        for run1, run2 in combinations(RUNS, 2):
            matched, summary, unmatched1, unmatched2, similarity = match_concepts_between_runs(
                all_concepts[dataset][setting][run1], all_concepts[dataset][setting][run2], top_k=MATCHING_TOP_K)
            matched_rows.extend(matched.assign(dataset=dataset, setting=setting, run1=run1, run2=run2).to_dict("records"))
            summary_rows.append(dict(dataset=dataset, setting=setting, run1=run1, run2=run2,
                unmatched_run1=json.dumps(unmatched1), unmatched_run2=json.dumps(unmatched2), **summary))
matched_results = pd.DataFrame(matched_rows)
matching_summary = pd.DataFrame(summary_rows)
matched_results.to_csv(OUTPUT_DIR / "matched_concepts.csv", index=False)
matching_summary.to_csv(OUTPUT_DIR / "matched_jaccard_by_run_pair.csv", index=False)
display(matching_summary.groupby(["dataset", "setting"])["mean_matched_jaccard"].agg(["mean", "std"]))


In [ ]:
if RUN_PLOTS:
    out_dir = str(OUTPUT_DIR)
    zeisel = matching_summary[(matching_summary.dataset == "Zeisel") & (matching_summary.setting == "Original")].mean_matched_jaccard.to_numpy()
    pollen = matching_summary[(matching_summary.dataset == "Pollen") & (matching_summary.setting == "Original")].mean_matched_jaccard.to_numpy()
    data = [zeisel, pollen]
    labels = ['Zeisel', 'Pollen']
    means = [x.mean() for x in data]
    stds = [x.std(ddof=1) for x in data]
    colors = ['#4C78A8', '#F58518']
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 9, 'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'axes.linewidth': 0.8, 'pdf.fonttype': 42, 'ps.fonttype': 42})
    (fig, ax) = plt.subplots(figsize=(3.4, 3.0))
    x = np.arange(len(labels))
    bars = ax.bar(x, means, width=0.58, color=colors, alpha=0.65, edgecolor=colors, linewidth=0.9, zorder=1)
    ax.errorbar(x, means, yerr=stds, fmt='none', ecolor='black', elinewidth=1.0, capsize=4, capthick=1.0, zorder=3)
    for (i, values) in enumerate(data):
        jitter = np.linspace(-0.11, 0.11, len(values))
        ax.scatter(x[i] + jitter, values, s=22, facecolor='white', edgecolor=colors[i], linewidth=1.0, zorder=4)
    for (i, (mean, std)) in enumerate(zip(means, stds)):
        ax.text(x[i], mean + std + 0.085, f'{mean:.3f} ± {std:.3f}', ha='center', va='bottom', fontsize=8.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Matched concept Jaccard similarity')
    ax.set_ylim(0, 0.82)
    ax.set_yticks(np.arange(0, 0.81, 0.2))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out', length=3, width=0.8)
    plt.tight_layout()
    pdf_path = os.path.join(out_dir, 'LLM_run_Jaccard_reproducibility.pdf')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    print('Zeisel:', f'{means[0]:.3f} ± {stds[0]:.3f}')
    print('Pollen:', f'{means[1]:.3f} ± {stds[1]:.3f}')
    print('Saved PDF:', pdf_path)


## 5. Compare deposited GPT-5 and Claude scores

Match top-100 coherence ratings by dataset and exact concept name across the 16 benchmarks. Check unique names and complete correspondence between raters. Export agreement, agreement within one score point, Pearson and Spearman correlations, and the original count-bubble plot. These scores refer to the benchmark concept sets, rather than the five-run robustness sets.


In [ ]:
score_rows = []
coverage_rows = []
for path in sorted((INPUT_DIR / "scores").glob("GPT5_*_coherence_top100.json")):
    dataset = path.name[len("GPT5_"):-len("_coherence_top100.json")]
    gpt = json.loads(path.read_text())
    claude = json.loads((path.parent / f"Claude_{dataset}_coherence_top100.json").read_text())
    g = {r["concept_name"]: r["coherence_score"] for r in gpt["scores"]}
    c = {r["concept_name"]: r["coherence_score"] for r in claude["scores"]}
    if len(g) != len(gpt["scores"]) or len(c) != len(claude["scores"]) or set(g) != set(c):
        raise ValueError(f"Non-unique or unmatched scored concepts: {dataset}")
    reference = load_saved_concepts(CONCEPT_DIR / f"GPT5_{dataset}_concepts.json")
    reference_names = {item["name"] for item in reference}
    coverage_rows.append(dict(dataset=dataset, gpt_n=len(g), claude_n=len(c), reference_n=len(reference),
        names_match_reference=set(g) == reference_names,
        only_in_scores=json.dumps(sorted(set(g) - reference_names)),
        only_in_reference=json.dumps(sorted(reference_names - set(g)))))
    for name in sorted(g):
        if g[name] not in range(1, 6) or c[name] not in range(1, 6):
            raise ValueError(f"Invalid coherence score: {dataset}, {name}")
        score_rows.append(dict(dataset=dataset, concept_name=name, gpt5_score=g[name], claude_opus5_score=c[name]))
df_compare = pd.DataFrame(score_rows)
df_compare["score_difference"] = df_compare.claude_opus5_score - df_compare.gpt5_score
df_compare["absolute_difference"] = df_compare.score_difference.abs()
df_compare.to_csv(OUTPUT_DIR / "GPT5_Claude_concept_scores.csv", index=False)
pd.DataFrame(coverage_rows).to_csv(OUTPUT_DIR / "score_coverage.csv", index=False)
pearson_r, pearson_p = pearsonr(df_compare.gpt5_score, df_compare.claude_opus5_score)
spearman_r, spearman_p = spearmanr(df_compare.gpt5_score, df_compare.claude_opus5_score)
agreement_statistics = dict(n_datasets=df_compare.dataset.nunique(), n_concepts=len(df_compare),
    exact_agreement=float((df_compare.absolute_difference == 0).mean()),
    within_one=float((df_compare.absolute_difference <= 1).mean()),
    mean_GPT5=float(df_compare.gpt5_score.mean()), mean_Claude=float(df_compare.claude_opus5_score.mean()),
    mean_difference=float(df_compare.score_difference.mean()), mean_absolute_difference=float(df_compare.absolute_difference.mean()),
    pearson_r=float(pearson_r), pearson_p=float(pearson_p), spearman_r=float(spearman_r), spearman_p=float(spearman_p))
(OUTPUT_DIR / "GPT5_Claude_agreement.json").write_text(json.dumps(agreement_statistics, indent=2))
display(pd.Series(agreement_statistics))


In [ ]:
if RUN_PLOTS:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    agreement = (df_compare['gpt5_score'] == df_compare['claude_opus5_score']).mean()
    within1 = ((df_compare['gpt5_score'] - df_compare['claude_opus5_score']).abs() <= 1).mean()
    scores = [1, 2, 3, 4, 5]
    count_df = df_compare.groupby(['gpt5_score', 'claude_opus5_score']).size().reset_index(name='count')
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 12, 'axes.linewidth': 1.2, 'pdf.fonttype': 42, 'ps.fonttype': 42})
    (fig, ax) = plt.subplots(figsize=(5.2, 5.2))
    sc = ax.scatter(count_df['gpt5_score'], count_df['claude_opus5_score'], s=count_df['count'] * 90, c=count_df['count'], cmap='Blues', edgecolors='black', linewidths=0.8)
    for (_, row) in count_df.iterrows():
        ax.text(row['gpt5_score'], row['claude_opus5_score'], str(int(row['count'])), ha='center', va='center', fontsize=10, color='black')
    ax.plot([1, 5], [1, 5], '--', color='gray', lw=1.5)
    ax.set_xlim(0.5, 5.5)
    ax.set_ylim(0.5, 5.5)
    ax.set_xticks(scores)
    ax.set_yticks(scores)
    ax.set_xlabel('GPT-5 coherence score')
    ax.set_ylabel('Claude Opus-5 coherence score')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.text(0.98, 0.02, f'Agreement = {agreement * 100:.1f}%\n|Δ| ≤ 1 = {within1 * 100:.1f}%', transform=ax.transAxes, ha='right', va='bottom', fontsize=11)
    cbar = plt.colorbar(sc)
    cbar.set_label('Number of concepts')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'LLM_agreement_bubble.pdf', dpi=600, bbox_inches='tight')
    plt.savefig(OUTPUT_DIR / 'LLM_agreement_bubble.png', dpi=600, bbox_inches='tight')
    plt.show()


## 6. Optional Claude coherence evaluation

The deposited scores are the reference outputs. To obtain new scores, set `RUN_NEW_CLAUDE_SCORING=True`, install the Anthropic SDK, and provide `ANTHROPIC_API_KEY`. `CLAUDE_MODEL` can override the historical model identifier recorded in the scores, `claude-opus-5`. This optional section sends the selected gene lists to the configured provider and writes new outputs to a separate directory. Hosted-model outputs may differ from the reference outputs.

The evaluation uses the same coherence prompt, maximum output length of 300 tokens, and disabled thinking as the supplied analysis. No seed was passed to Claude. Full prompts, raw text responses, returned model identifiers, and query times are saved with new scores.


In [ ]:
CLAUDE_MODEL = os.environ.get("CLAUDE_MODEL", "claude-opus-5")
CLAUDE_TOP_K = 100
CLAUDE_DATASETS = [r["dataset"] for r in coverage_rows]

def score_concepts_with_claude(client, concepts, dataset, out_dir):
    from datetime import datetime, timezone
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    raw_responses = []
    for concept in concepts:
        genes = concept["genes"][:CLAUDE_TOP_K]
        prompt = PROMPTS["Concept coherence"].format(concept_name=concept["name"], gene_list=", ".join(genes))
        if genes:
            response = client.messages.create(model=CLAUDE_MODEL, max_tokens=300,
                thinking={"type": "disabled"}, system="Return ONLY valid JSON. No explanations.",
                messages=[{"role": "user", "content": prompt}])
            text = "\n".join(block.text for block in response.content if block.type == "text").strip()
            parsed_text = text
            if parsed_text.startswith("```"):
                parsed_text = parsed_text.strip("`").replace("json", "").strip()
            score = int(json.loads(parsed_text)["coherence_score"])
            if score not in range(1, 6):
                raise ValueError(f"Invalid coherence score: {concept['name']}")
            raw_responses.append(dict(concept_name=concept["name"], prompt=prompt, response_text=text,
                response=response.model_dump(mode="json"), queried_at=datetime.now(timezone.utc).isoformat()))
        else:
            score = 0
        rows.append(dict(concept_name=concept["name"], num_genes=len(genes), coherence_score=score,
                         source_topics=concept.get("source_topics", [])))
        output = dict(method="concept", dataset=dataset, top_k=CLAUDE_TOP_K, num_concepts=len(concepts),
                      model=CLAUDE_MODEL, scores=rows)
        (out_dir / f"Claude_{dataset}_coherence_top{CLAUDE_TOP_K}.json").write_text(json.dumps(output, indent=2))
        (out_dir / f"Claude_{dataset}_raw_responses.json").write_text(json.dumps(raw_responses, indent=2))
    return output

if RUN_NEW_CLAUDE_SCORING:
    from anthropic import Anthropic
    client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    for dataset in CLAUDE_DATASETS:
        concepts = load_saved_concepts(CONCEPT_DIR / f"GPT5_{dataset}_concepts.json")
        score_concepts_with_claude(client, concepts, dataset, OUTPUT_DIR / "new_claude_scores")


## Output guide

Paths below are relative to `outputs/robustness`.

| Analysis | Main output |
|---|---|
| All 30 runs and two consensus evaluations | `concept_performance.csv`, `performance_summary.csv` |
| Saved run concept names and genes | `run_concepts.csv` |
| Per-run cell assignments and scores | `<dataset>/<setting>/run<run>/assignments.csv`, `concept_scores.npz` |
| Run stability with consensus | `Zeisel_Pollen_LLM_run_stability_with_consensus.pdf` |
| Prompt sensitivity and concept counts | `<dataset>/<dataset>_prompt_sensitivity.pdf`, `<dataset>/<dataset>_prompt_concept_number.pdf` |
| Matched concepts and run-pair similarities | `matched_concepts.csv`, `matched_jaccard_by_run_pair.csv` |
| Original-prompt Jaccard comparison | `LLM_run_Jaccard_reproducibility.pdf` |
| Deposited GPT-5/Claude score comparison | `GPT5_Claude_concept_scores.csv`, `GPT5_Claude_agreement.json`, `score_coverage.csv` |
| GPT-5/Claude agreement plot | `LLM_agreement_bubble.pdf` |
| Optional new Claude scores and raw responses | `new_claude_scores/` |

The final cell saves input checksums, scoring parameters, and package versions.


## 7. Save run settings


In [ ]:
manifest = json.loads((INPUT_DIR / "manifest.json").read_text())
configuration = dict(python=platform.python_version(),
    packages={p: importlib.metadata.version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib"]},
    inputs=dict(mat_dir=str(MAT_DIR), gmt_paths={s: str(p) for s, p in GMT_PATHS.items()}),
    generation_seed=1, run_ids=RUNS, scoring_top_k=TOP_K, matching_top_k=MATCHING_TOP_K,
    plotted_information_metric=INFO_METRIC, stability_tc_background=STABILITY_TC_BACKGROUND,
    prompt_tc_background=PROMPT_TC_BACKGROUND, new_claude_scoring=RUN_NEW_CLAUDE_SCORING,
    copied_input_sha256={r["copy"]: hashlib.sha256((TUTORIAL_DIR / r["copy"]).read_bytes()).hexdigest()
                         for r in manifest if "sha256" in r},
    prompts_sha256=hashlib.sha256((INPUT_DIR / "prompts.json").read_bytes()).hexdigest())
configuration["package_paths"] = runtime_paths()
(OUTPUT_DIR / "run_configuration.json").write_text(json.dumps(configuration, indent=2))
print("Outputs:", OUTPUT_DIR)
